本地使用一张图(小狗图片)，尝试进行一下zero shot classification图像分类

In [1]:
import torch
from PIL import Image
from transformers import AutoProcessor, AutoModel

# 1. 指向你本地的文件夹路径
model_path = "../../models/AI-ModelScope/chinese-clip-vit-base-patch16"

# 2. 加载模型和处理器
# trust_remote_code=True 是必须的，因为 Chinese-CLIP 有自定义的模型结构
model = AutoModel.from_pretrained(model_path, trust_remote_code=True)
processor = AutoProcessor.from_pretrained(model_path, trust_remote_code=True)

model.eval() # 切换到推理模式

# 3. 准备图像和中文标签
image = Image.open("dog.jpg")
# 这里可以写非常地道的中文
candidate_labels = ["一张小狗的照片", "一张小猫的照片", "一张汽车的照片", "一张哈士奇的特写"]

# 4. 预处理输入
inputs = processor(text=candidate_labels, images=image, return_tensors="pt", padding=True)

# 5. 推理计算
with torch.no_grad():
    outputs = model(**inputs)
    # 获取图像和文本的相似度
    logits_per_image = outputs.logits_per_image 
    probs = logits_per_image.softmax(dim=1) # 转化为百分比概率

# 6. 可视化结果
print("--- 零样本分类结果 ---")
for i, label in enumerate(candidate_labels):
    print(f"标签: {label:10s} | 概率: {probs[0][i].item():.4%}")

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


--- 零样本分类结果 ---
标签: 一张小狗的照片    | 概率: 99.5420%
标签: 一张小猫的照片    | 概率: 0.2353%
标签: 一张汽车的照片    | 概率: 0.0024%
标签: 一张哈士奇的特写   | 概率: 0.2203%
